## 0. Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from math import sqrt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, accuracy_score
import tensorflow as tf
from keras.models import Model
from keras.layers import (Input, Conv2D, MaxPooling2D, Reshape, Dense, Embedding,
                                     Bidirectional, LSTM, AdditiveAttention, Concatenate,
                                     BatchNormalization, Activation)
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

## 1. Read dataset

In [ ]:
images = np.load("../datasets/images.npy", mmap_mode="r")
labels = np.load("../datasets/labels.npy")

max_len = max(len(str(label)) for label in labels)

print("Number of images:", len(images))
print("Image shape:", images.shape)
print("Maximum label length:", max_len)

In [ ]:
IMG_SHAPE = (28, 224, 1)
MAX_LABEL_LENGTH = 8  
MAX_OUTPUT_LENGTH = MAX_LABEL_LENGTH + 1

PAD_TOKEN = 10
START_TOKEN = 11
END_TOKEN = 12

VOCAB_SIZE = 13  # 0-9, PAD, START, END

VERSION = "v1"  # ĐỔI TÊN VERSION Ở ĐÂY VỚI MỖI KIẾN TRÚC KHÁC NHAU

## 2. Preprocessing

In [ ]:
# Chuyển label từ chuỗi thành danh sách các chữ số
seq = np.full((len(labels), MAX_OUTPUT_LENGTH + 1), PAD_TOKEN, dtype=np.int64)
for i, label in enumerate(labels):
    digits = [int(d) for d in str(label)]
    seq[i, 0] = START_TOKEN
    seq[i, 1 : len(digits) + 1] = digits
    seq[i, len(digits) + 1] = END_TOKEN

dec_in = seq[:, :-1]
dec_out = seq[:, 1:]

In [ ]:
print("Maximum digit length:", MAX_LABEL_LENGTH)
print("Target length:", MAX_OUTPUT_LENGTH)

print("\nExample labels:")
for i in range(3):
    print("Original:", labels[i])
    print("Decoder input:", dec_in[i])
    print("Decoder target:", dec_out[i])
    print()

## 3. Data Split

Chia train/val/test = 70/15/15

In [ ]:
idx = np.arange(len(images))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=42)

X_train, X_val, X_test = images[idx_train], images[idx_val], images[idx_test]
Din_train, Din_val, Din_test = dec_in[idx_train], dec_in[idx_val], dec_in[idx_test]
Dout_train, Dout_val, Dout_test = dec_out[idx_train], dec_out[idx_val], dec_out[idx_test]
y_train, y_val, y_test = labels[idx_train], labels[idx_val], labels[idx_test]

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

In [ ]:
plt.figure(figsize=(8, 2))
plt.imshow(X_train[2].squeeze(), cmap='gray')
plt.title(f"Label: {y_train[2]}")
plt.axis('off')
plt.show()

## 4. Train model

In [ ]:
units = 256

# Encoder (CNN + BiLSTM)
img_input = Input(shape=IMG_SHAPE)                                       # (28, 224, 1)
x = Conv2D(32, (3, 3), padding='same', use_bias=False)(img_input)        # (28, 224, 32)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D((2, 2))(x)                                              # (14, 112, 32)
x = Conv2D(64, (3, 3), padding='same', use_bias=False)(x)                # (14, 112, 64)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D((2, 2))(x)                                              # (7, 56, 64)
x = Conv2D(128, (3, 3), padding='same', use_bias=False)(x)               # (7, 56, 128)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D((7, 1))(x)                                              # (1, 56, 128): chỉ gộp chiều cao, giữ 56 cột

# Ảnh gồm 8 ô rộng 28px, sau CNN mỗi ô còn 7 cột -> 56 cột = 8 ô x 7 cột
x = Reshape((8, 7 * 128))(x)                                             # (8, 896)

enc_out, f_h, f_c, b_h, b_c = Bidirectional(LSTM(units // 2, return_sequences=True, return_state=True))(x)
state_h = Concatenate()([f_h, b_h])                                      # (256,)
state_c = Concatenate()([f_c, b_c])                                      # (256,)

# Decoder (LSTM + Additive Attention)
dec_input = Input(shape=(MAX_OUTPUT_LENGTH,))
emb = Embedding(VOCAB_SIZE, 64)(dec_input)
dec_out = LSTM(units, return_sequences=True)(emb, initial_state=[state_h, state_c])

context = AdditiveAttention()([dec_out, enc_out])
merged = Concatenate()([dec_out, context])
z = Dense(256, activation='relu')(merged)
outputs = Dense(VOCAB_SIZE, activation='softmax')(z)

In [ ]:
model = Model(inputs=[img_input, dec_input], outputs=outputs)
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', weighted_metrics=['sparse_categorical_accuracy'])
model.summary()

In [ ]:
# Mask những vị trí PAD
train_mask = (Dout_train != PAD_TOKEN).astype('float64')
val_mask = (Dout_val != PAD_TOKEN).astype('float64')

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# Training
history = model.fit([X_train, Din_train], Dout_train, 
                    sample_weight=train_mask, 
                    epochs=50, 
                    batch_size=64, 
                    verbose=1, 
                    validation_data=([X_val, Din_val], Dout_val, val_mask), 
                    callbacks=[early_stop, reduce_lr])

## 5. Prediction

In [ ]:
# khi dự đoán không có nhãn thật nên phải sinh từng chữ số một,
# chữ số vừa sinh ra được đưa vào làm input cho bước kế tiếp

# Khởi tạo Test decoder input với toàn bộ là PAD_TOKEN, riêng vị trí đầu tiên là START_TOKEN
n_samples = len(X_test)
test_dec_in = np.full((n_samples, MAX_OUTPUT_LENGTH), PAD_TOKEN, dtype=np.int64)
test_dec_in[:, 0] = START_TOKEN

for t in range(MAX_OUTPUT_LENGTH):
    preds = model.predict([X_test, test_dec_in], batch_size=256, verbose=0)

    # Chỉ cho phép dự đoán digit hoặc END
    logits = preds[:, t, :].copy()
    logits[:, PAD_TOKEN] = -1
    logits[:, START_TOKEN] = -1
    next_token = np.argmax(logits, axis=-1)

    if t + 1 < MAX_OUTPUT_LENGTH:
        test_dec_in[:, t + 1] = next_token

outputs = []
for seq in test_dec_in[:, 1:]:
    # Lấy các token từ vị trí 1 đến khi gặp END_TOKEN hoặc hết độ dài
    output = []
    for token in seq:
        if token >= 10:
            break
        output.append(str(token))
    outputs.append(float(''.join(output)) if output else 0.0)

y_pred = np.array(outputs, dtype=np.float64)

## 6. Evaluation

In [ ]:
y_true_num = np.array(y_test, dtype=np.float64)
y_pred_num = np.array(y_pred, dtype=np.float64)

mae     = mean_absolute_error(y_true_num, y_pred_num)
rmse    = np.sqrt(mean_squared_error(y_true_num, y_pred_num))
r2      = r2_score(y_true_num, y_pred_num)
nse_denom   = np.sum((y_true_num - np.mean(y_true_num)) ** 2)
nse     = 1.0 - np.sum((y_true_num - y_pred_num) ** 2) / nse_denom if nse_denom else np.nan
acc     = accuracy_score(y_true_num, y_pred_num)

print(f"Sequence Accuracy: {acc:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")
print(f"NSE:  {nse:.4f}")

### Save Result

In [ ]:
output_path = "../output/HuyModel/"  
csv_path = output_path + "huy_model_results.csv"   
model_path = output_path + f"huy_model_{VERSION}.keras"   

import os
os.makedirs(output_path, exist_ok=True)

model.save(model_path)  

results = {
    "Model": f"Huy Model {VERSION}",     
    "Version": VERSION,
    "Sequence_Accuracy": acc,
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "NSE": nse
}

results_df = pd.DataFrame([results])

if os.path.exists(csv_path):
    results_df = pd.concat([pd.read_csv(csv_path), results_df], ignore_index=True)
results_df.to_csv(csv_path, index=False)

print(f"Model saved: {model_path}")
print(f"Results saved: {csv_path}")